# اجرای ویترین‌یاب در Google Colab
این نوت‌بوک Laravel و FastAPI را بدون Docker اجرا می‌کند و در پایان یک لینک HTTPS موقت نمایش می‌دهد. چون مخزن Public است، به GitHub Token نیاز ندارید.

In [ ]:
# Clone the public repository
import os, shutil, subprocess
REPOSITORY = 'https://github.com/Alirezaab78/clothes_search.git'
PROJECT_DIR = '/content/clothes_search'
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, PROJECT_DIR], check=True)
print('✅ Project cloned:', PROJECT_DIR)

In [ ]:
%%bash
set -euo pipefail
PROJECT_DIR=/content/clothes_search
export DEBIAN_FRONTEND=noninteractive

# ۱. نصب پیش‌نیازهای لینوکسی و PHP
apt-get update -qq
apt-get install -y -qq php-cli php-curl php-mbstring php-xml php-zip php-sqlite3 unzip curl git
if ! command -v composer >/dev/null; then
  curl -fsSL https://getcomposer.org/installer -o /tmp/composer-setup.php
  php /tmp/composer-setup.php --install-dir=/usr/local/bin --filename=composer --quiet
fi

# ۲. نصب نیازمندی‌های هوش مصنوعی و اجرای FastAPI
cd $PROJECT_DIR/ai-service
pip install -r requirements.txt --quiet --no-warn-conflicts
nohup uvicorn app.main:app --host 127.0.0.1 --port 8001 >/tmp/fashion-ai.log 2>&1 &

# ۳. راه‌اندازی لاراول و SQLite
cd $PROJECT_DIR/laravel-app
composer install --no-interaction --prefer-dist --optimize-autoloader --ignore-platform-reqs -q
cp -n .env.example .env || true
touch database/database.sqlite
sed -i 's|^DB_CONNECTION=.*|DB_CONNECTION=sqlite|' .env
sed -i 's|^FASHION_AI_URL=.*|FASHION_AI_URL=http://127.0.0.1:8001|' .env
grep -q '^FASHION_AI_URL=' .env || echo 'FASHION_AI_URL=http://127.0.0.1:8001' >> .env
php artisan key:generate --force
php artisan migrate --force
php artisan storage:link || true
nohup php artisan serve --host=127.0.0.1 --port=8000 >/tmp/laravel.log 2>&1 &

# ۴. بررسی بالا آمدن سرورها
echo 'Waiting for services to spin up...'
for i in $(seq 1 120); do curl -fs http://127.0.0.1:8001/health >/dev/null 2>&1 && break; sleep 1; done
curl -fs http://127.0.0.1:8001/health >/dev/null
for i in $(seq 1 30); do curl -fs http://127.0.0.1:8000 >/dev/null 2>&1 && break; sleep 1; done
curl -fs http://127.0.0.1:8000 >/dev/null
echo '✅ All services (FastAPI & Laravel) are running successfully!'


In [ ]:
%%bash
set -euo pipefail
# دانلود و نصب کلاینت Cloudflare Tunnel
if [ ! -f /usr/local/bin/cloudflared ]; then
  curl -fsSL -L --retry 3 https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
  chmod +x /usr/local/bin/cloudflared
fi
pkill cloudflared || true
nohup cloudflared tunnel --url http://127.0.0.1:8000 >/tmp/cloudflared.log 2>&1 &
for i in $(seq 1 40); do
  URL=$(grep -oE 'https://[a-zA-Z0-9-]+\.trycloudflare\.com' /tmp/cloudflared.log | head -n 1 || true)
  if [ -n "$URL" ]; then break; fi
  sleep 2
done
if [ -z "$URL" ]; then
  echo 'Tunnel URL was not created. Log:'
  cat /tmp/cloudflared.log
  exit 1
fi
echo '🎉 سایت شما با موفقیت آنلاین شد!'
echo "🔗 لینک دسترسی: $URL"


## استفاده
در یک Runtime تازه، سه سلول کد را به ترتیب یا با Runtime → Run all اجرا کنید. لینک trycloudflare.com در خروجی سلول آخر نشان داده می‌شود. این لینک با پایان Runtime نامعتبر می‌شود.

In [ ]:
%%bash
set -euo pipefail
PROJECT_DIR=/content/clothes_search/laravel-app
mkdir -p $PROJECT_DIR/storage/framework/{sessions,views,cache,testing}
mkdir -p $PROJECT_DIR/storage/logs $PROJECT_DIR/bootstrap/cache
chmod -R 777 $PROJECT_DIR/storage $PROJECT_DIR/bootstrap/cache
cd $PROJECT_DIR
php artisan config:clear
php artisan view:clear
php artisan cache:clear
pkill -f 'artisan serve' || true
nohup php artisan serve --host=127.0.0.1 --port=8000 >/tmp/laravel.log 2>&1 &
sleep 2
echo '✅ کش‌ها ساخته شدند و سرور لاراول مجدداً ری‌استارت شد!'
